# Demo — Fine-Grained Access with Lake Formation (live AWS)

**Trailhead Provisions** must let a marketing analyst work with customer data **without seeing
contact PII**. This walkthrough demonstrates the *technique*: make a **column-level Lake
Formation grant**, then **verify it by querying as that persona in Athena**. The exercise asks
you to configure all four personas and run the full verification.

> Runs against live AWS Lake Formation + Athena when provisioned; falls back to an equivalent
> local enforcement engine offline — the grant model and outcomes are identical.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)
print("Personas:", list(PERSONAS))
gc.grants_summary()

## Grant column-level access
The analyst should see the customer table **without** the `email` and `phone` PII columns. Under the live backend this is a real Lake Formation column grant.

In [ ]:
gc.grant_columns("co_marketing_analyst", "customer", exclude=["email", "phone"])
gc.grants_summary()

## Verify by querying as the persona
`query_as` assumes the persona role and runs the query through Athena (live) so Lake Formation enforcement is what you see.

In [ ]:
df = gc.query_as("co_marketing_analyst", "SELECT * FROM customer LIMIT 5")
print("Columns visible to the analyst:", list(df.columns))   # email, phone are absent
df

The analyst's query succeeds but the PII columns are simply not there — enforced by
Lake Formation, not by asking the analyst to behave.

**Takeaway / your turn:** that's one persona, one LF mode. In the exercise you'll configure all
four — column, row, tag-based, and metadata-only — and run the verification that proves each
persona sees exactly what policy allows.